# Part 8 · ECoRe innovations 1–3

This notebook freezes the Part 7 encoder and tests three claims on source-heldout evidence: source-balanced author distributions, environment-relative geometry, and author-heldout episodic transfer. It does not rebuild Part 7 embeddings and does not train a production encoder.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, shutil, subprocess, sys
import pandas as pd

drive.mount('/content/drive')
REPO = Path('/content/drive/MyDrive/style_matching')
os.chdir(REPO)
EXP = REPO / 'artifacts/source_expansion_v2'
HELDOUT = REPO / 'data/all/meta/all_source_heldout_splits.parquet'
EMBEDDINGS = EXP / 'expanded_source_heldout_eval'
OUT = EXP / 'ecore_innovations_v1'
assert HELDOUT.exists(), f'Run Parts 1–7 first; missing {HELDOUT}'
for name in ('style_embedding_train_embeddings.npy', 'style_embedding_eval_embeddings.npy', 'style_embedding_scores.npz'):
    assert (EMBEDDINGS / name).exists(), f'Part 7 embedding artifact missing: {EMBEDDINGS / name}'
assert (REPO / 'scripts/evaluate_ecore_innovations.py').exists(), 'Pull the commit containing the redesigned Part 8 script'

def run(cmd):
    print('>>>', ' '.join(map(str, cmd)), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f'command failed with exit {code}: {" ".join(map(str, cmd))}')

# Remove only interrupted legacy Part 8 model directories. Completed checkpoints are preserved.
legacy_root = EXP / 'top4_full_retrain'
if legacy_root.exists():
    for candidate in legacy_root.glob('*/finetuned_authorship_expanded'):
        if not (candidate / 'training_config.json').exists():
            print('REMOVE interrupted legacy checkpoint:', candidate)
            shutil.rmtree(candidate)
    for model_name in ('strict_eval_finetuned_authorship', 'production_full_source_finetuned_authorship'):
        for candidate in legacy_root.glob(f'*/{model_name}'):
            if not (candidate / 'training_config.json').exists():
                print('REMOVE interrupted checkpoint:', candidate)
                shutil.rmtree(candidate)

print('Frozen input:', HELDOUT)
print('Reusing Part 7 embeddings:', EMBEDDINGS)
print('No encoder batches or production retraining will run in Part 8.')

## 8.1–8.3 · Falsifiable tests

Innovation 1 compares one centroid, hard prototype, and source-balanced soft prototypes. Innovation 2 uses language × register cohort-centred residual geometry and a shuffled-environment control. Innovation 3 trains a candidate-identity-free pairwise scorer on variable support episodes and evaluates it by whole-author cross-fitting.

In [ ]:
run([
    sys.executable, 'scripts/evaluate_ecore_innovations.py',
    '--input', str(HELDOUT),
    '--embedding-dir', str(EMBEDDINGS),
    '--output-dir', str(OUT),
    '--temperature', '0.08',
    '--author-folds', '5',
    '--hard-negatives', '12',
    '--bootstrap-runs', '5000',
    '--train-cap', '300',
    '--embedding-seed', '20260701',
    '--seed', '20260725',
])

report_path = OUT / 'ecore_innovation_metrics.json'
report = json.loads(report_path.read_text())
summary = pd.DataFrame([
    {
        'innovation': 1,
        'claim': 'author as a source-balanced distribution',
        **report['innovation_1_distributional_profile']['paired_profile_bootstrap'],
        'supported': report['innovation_1_distributional_profile']['supported'],
    },
    {
        'innovation': 2,
        'claim': 'environment-relative evidence',
        **report['innovation_2_cohort_relative_geometry']['paired_profile_bootstrap'],
        'supported': report['innovation_2_cohort_relative_geometry']['supported'],
    },
    {
        'innovation': 3,
        'claim': 'whole-author episodic transfer',
        **report['innovation_3_episodic_transfer']['paired_profile_bootstrap'],
        'supported': report['innovation_3_episodic_transfer']['supported'],
    },
])
display(summary)
display(pd.DataFrame(report['test_metrics']).T.sort_values('mrr', ascending=False))
display(pd.DataFrame(report['innovation_3_episodic_transfer']['variable_support_test_metrics']).T)
print('RETURN:', report_path)
print('RETURN:', OUT / 'ecore_innovation_scores.npz')

## 8.4 · Gutenberg-eligible author expansion

This stage scans Gutendex for single-author, plaintext, public-domain books and admits only author-language profiles with at least three independent works. It is restart-safe. Existing registry rows are preserved; the public Author Library is filtered later from the completed index.

In [ ]:
GUTENBERG = EXP / 'gutenberg_all_v1'
GUTENBERG.mkdir(parents=True, exist_ok=True)
CANDIDATES = GUTENBERG / 'eligible_authors_after1800.csv'
LANGUAGES = ('en', 'de', 'es', 'fr', 'it', 'ja', 'pl', 'ru', 'zh')
MAX_NEW_AUTHORS_PER_LANGUAGE = 0  # 0 = all qualifying authors; reruns resume downloaded sources

if CANDIDATES.exists():
    print('REUSE completed Gutenberg candidate scan:', CANDIDATES)
else:
    run([sys.executable, 'scripts/discover_gutenberg_authors.py',
         *sum((['--language', language] for language in LANGUAGES), []),
         '--min-works', '3', '--earliest-author-year', '1800',
         '--output', str(CANDIDATES)])
for language in LANGUAGES:
    run([sys.executable, 'scripts/fetch_gutendex.py', '--corpus', 'literary',
         '--language', language, '--registry', str(CANDIDATES),
         '--min-works', '3', '--max-works', '6',
         '--max-authors', str(MAX_NEW_AUTHORS_PER_LANGUAGE), '--skip-covered'])
print('RETURN:', CANDIDATES)

## 8.5 · Rebuild frozen evidence once

The encoder remains frozen. New texts receive embeddings; previously cached chunks are reused. Source-heldout splits are rebuilt because the candidate universe changed.

In [ ]:
CHUNKS_V3 = REPO / 'data/all/meta/all_sources_chunks.parquet'
COVERAGE_V3 = GUTENBERG / 'coverage.json'
HELDOUT_V3 = GUTENBERG / 'source_heldout_splits.parquet'
HELDOUT_REPORT_V3 = GUTENBERG / 'source_heldout_report.json'
EVAL_V3 = GUTENBERG / 'frozen_encoder_eval'
INDEX_V3 = GUTENBERG / 'index'
MODEL = REPO / 'artifacts/multilingual_author_style_v1'

run([sys.executable, 'scripts/build_chunk_parquet_from_sources.py', '--corpus', 'both',
     '--output', str(CHUNKS_V3), '--coverage-output', str(COVERAGE_V3),
     '--min-sources', '3', '--min-chunks', '30'])
run([sys.executable, 'scripts/make_source_heldout_splits.py', '--input', str(CHUNKS_V3),
     '--output', str(HELDOUT_V3), '--report', str(HELDOUT_REPORT_V3)])
run([sys.executable, 'scripts/style_embedding_recall.py', '--input', str(HELDOUT_V3),
     '--out-dir', str(EVAL_V3), '--model-name', str(MODEL), '--batch-size', '128',
     '--train-cap', '300', '--eval-splits', 'dev,test', '--device', 'cuda',
     '--reuse-input', str(HELDOUT), '--reuse-embedding-dir', str(EMBEDDINGS), '--skip-existing'])
run([sys.executable, 'scripts/multilingual_style_index.py', 'build', '--input', str(CHUNKS_V3),
     '--out-dir', str(INDEX_V3), '--model-name', str(MODEL),
     '--topic-model-name', 'intfloat/multilingual-e5-base',
     '--embedding-cache', str(INDEX_V3 / 'chunk_embeddings.npz'),
     '--topic-embedding-cache', str(INDEX_V3 / 'topic_chunk_embeddings.npz'),
     '--batch-size', '128', '--per-source-cap', '50', '--profile-cap', '600',
     '--profile-strategy', 'single_centroid', '--heldout-report', str(HELDOUT_REPORT_V3),
     '--model-label', 'challenger_finetuned', '--artifact-version', 'gutenberg_all_v1',
     '--device', 'cuda'])

## 8.6 · Hubness punishment

Frequency and local-density corrections are tuned by source-grouped cross-fitting inside dev. Candidate penalties are standardized within language and source-support tier. Test is opened once; a correction is adopted only when its paired profile-bootstrap MRR interval is wholly positive and Recall@3 does not decrease.

In [ ]:
HUBNESS = GUTENBERG / 'hubness_correction'
run([sys.executable, 'scripts/evaluate_hubness_correction.py', '--input', str(HELDOUT_V3),
     '--scores', str(EVAL_V3 / 'style_embedding_scores.npz'), '--output-dir', str(HUBNESS),
     '--top-k', '10', '--folds', '5',
     '--lambdas', '0,0.02,0.05,0.08,0.1,0.15,0.2',
     '--bootstrap-runs', '5000', '--seed', '20260726'])
hub_report = json.loads((HUBNESS / 'hubness_correction_metrics.json').read_text())
display(pd.DataFrame({name: result['test'] for name, result in hub_report['methods'].items()}).T)
print('selected:', hub_report['selected'])
run([sys.executable, 'scripts/attach_hubness_correction.py', '--index-dir', str(INDEX_V3),
     '--report', str(HUBNESS / 'hubness_correction_metrics.json')])
print('RETURN:', HUBNESS / 'hubness_correction_metrics.json')

## 8.7 · Synchronize the public Author Library

Only authors present in the completed index are exported. Missing registry biographies are reported and must be enriched before those new names can appear publicly; research registry rows are never destroyed.

In [ ]:
LIBRARY_AUDIT = GUTENBERG / 'author_library_coverage.json'
run([sys.executable, 'scripts/export_author_library.py',
     '--profiles', str(INDEX_V3 / 'profiles.parquet'),
     '--coverage-output', str(LIBRARY_AUDIT)])
library_audit = json.loads(LIBRARY_AUDIT.read_text())
print(json.dumps({key: value if not isinstance(value, list) else len(value)
                  for key, value in library_audit.items()}, indent=2))
print('NEW INDEX:', INDEX_V3)
print('RETURN:', LIBRARY_AUDIT)

## Decision rule

A claim passes only when its paired author-profile bootstrap 95% interval is wholly above zero. Innovation 2 must additionally beat the shuffled-environment control. Unsupported claims remain negative results; Part 8 does not hide them by training a larger encoder or adding more indicators.